In [ ]:
# Visualise Bilby result

import bilby
from BASIS.models import base
import numpy as np
import matplotlib.pyplot as plt
from BASIS.modules import ringFitting
import ehtim as eh

def plot_beam(ax, beamparams, fov, labelc='k'):
    beamimage = eh.image.make_empty(imgdim, fov*1e-6/206265, ra=0, dec=0)
    beamparams = [beamparams[0], beamparams[1], beamparams[2],
                        -.35 * beamimage.fovx(), -.35 * beamimage.fovy()]
    beamimage = beamimage.add_gauss(1, beamparams)
    halflevel = 0.5 * np.max(beamimage.imvec)
    beamimarr = (beamimage.imvec).reshape(beamimage.ydim, beamimage.xdim)
    ax.contour(beamimarr, levels=[halflevel], colors=labelc, linewidths=1)

fov = 450 # Field of view (uas)
imgdim = 128 # Image dimension (pixels)


bilby_basedir = 'bilby_outdir/'
bilby_label = 'test_xsrggg'
model_list = ['xsringauss', 'gauss', 'gauss']
blur_beam = True
result = bilby.result.read_in_result(filename=f'{bilby_basedir}{bilby_label}/{bilby_label}_result.json')


model_params = result.posterior.median().to_dict()
print(model_params)
model_instance = base.BaseModel(params=model_params, model_list=model_list, dim=imgdim, fov=fov)

# measure key_params with errors
if model_instance.key_params() is not None:
    N = 500
    for i in range(N):
        sample = result.posterior.sample().to_dict()
        sample = {k: list(v.items())[0][1] for k, v in sample.items()}
        model_instance = base.BaseModel(params=sample, model_list=model_list, dim=imgdim, fov=fov)
        if i == 0:
            key_param_samples = {k: [] for k in model_instance.key_params()}
        for k in model_instance.key_params():
            key_param_samples[k].append(model_instance.key_params()[k])

    print('Key parameter mean and 1-sigma errors:')
    for k in model_instance.key_params():
        mean = np.mean(key_param_samples[k])
        std = np.std(key_param_samples[k])
        print(f'{k}: {mean:.4f} +/- {std:.4f}')


model_params = result.posterior.median().to_dict()
model_instance = base.BaseModel(params=model_params, model_list=model_list, dim=imgdim, fov=fov)

fig, ax = plt.subplots()
model_img = eh.image.make_empty(imgdim, fov * 1e-6/206265, 0, 0)
model_img._imdict['I'] = model_instance.sky_map().flatten()

if blur_beam:
    uvfits_path = '../data/uvfits/' + '_'.join(bilby_label.split('_')[:-1]) + '.uvfits'
    beamObs = eh.obsdata.load_uvfits(uvfits_path)
    model_img = model_img.blur_gauss(beamObs.fit_beam())
    ax.imshow(model_img.imvec.reshape(model_img.ydim, model_img.xdim), cmap='afmhot', interpolation='gaussian')
    plot_beam(ax, beamObs.fit_beam(), fov, labelc='white')
else:
    ax.imshow(model_instance.sky_map(), extent=[fov/2, -fov/2, -fov/2, fov/2], cmap='afmhot', interpolation='gaussian')



# # Ringfitting
# ringfitter = ringFitting.RingFit(model_img, interp_factor=2, blur_uas=0, total_flux=1)
# ringfitter.run_all(verbose=True)
# height_ratio = [6,3]
# fig, ax = plt.subplots(2,1, figsize=(np.sum(height_ratio),height_ratio[0]), height_ratios=height_ratio)
# plt.subplots_adjust(hspace=0.0, wspace=0.0)
# save_path = None
# ringfitter.plot_all(ax[0], ax[1], cmap='Greys', ratio=height_ratio, save_path=save_path)

N = 2
# visualise samples from posterior
fig, axs = plt.subplots(N, N, figsize=(10,10))
plt.subplots_adjust(hspace=0, wspace=0)
for i in range(N):
    for j in range(N):
        sample = result.posterior.sample().to_dict()
        sample = {k: list(v.items())[0][1] for k, v in sample.items()}
        model_instance = base.BaseModel(params=sample, model_list=model_list, dim=imgdim, fov=fov)
        axs[i,j].imshow(model_instance.sky_map(), extent=[fov/2, -fov/2, -fov/2, fov/2], cmap='afmhot', interpolation='gaussian')
        axs[i,j].axis('off')
plt.tight_layout()

# result.plot_corner()


In [ ]:
# Compare Bayesian evidence across model runs for the same dataset
import os
import glob
import bilby

dataset_key = "ER6_SgrA_2017_097_hilo_hops_netcal_StokesI_preimcal0"
bilby_basedir = "bilby_outdir"

result_files = sorted(glob.glob(os.path.join(bilby_basedir, "*", "*_result.json")))
matched_files = [f for f in result_files if dataset_key in os.path.basename(f)]

if not matched_files:
    print(f"No Bilby result files matched dataset_key='{dataset_key}'.")
else:
    rows = []
    for filename in matched_files:
        result = bilby.result.read_in_result(filename=filename)
        label = os.path.basename(filename).replace("_result.json", "").replace(dataset_key, "").strip("_")
        rows.append({
            "label": label,
            "log_evidence": float(result.log_evidence),
            "log_evidence_err": float(result.log_evidence_err),
            "log_noise_evidence": float(getattr(result, "log_noise_evidence", float("nan"))),
        })

    rows = sorted(rows, key=lambda r: r["log_evidence"], reverse=True)
    best_logz = rows[0]["log_evidence"]

    print(f"Matched {len(rows)} runs for dataset_key='{dataset_key}'")
    print(f"{'MODEL':20s} {'LOG_Z':>12s} {'ERR':>10s} {'dLOG_Z':>14s}")
    for r in rows:
        dlogz = r["log_evidence"] - best_logz
        print(f"{r['label'][:20]:20s} {r['log_evidence']:12.3f} {r['log_evidence_err']:10.3f} {dlogz:14.3f}")

